# The Heritage Short — Wan 2.2 TI2V-5B (2xT4 Save-As-You-Go)
Reads `storyboard.json` and keyframe PNGs from `/kaggle/input/heritage-short-inputs/` and saves each generated MP4 clip immediately to `/kaggle/working/outputs/shots/`.

In [ ]:
!pip install -q --upgrade diffusers transformers accelerate imageio imageio-ffmpeg ftfy sentencepiece
import glob, json, os, time
from pathlib import Path
import torch
from diffusers import WanImageToVideoPipeline
from diffusers.utils import export_to_video, load_image

sb_matches = list(Path('/kaggle/input').rglob('storyboard.json'))
if not sb_matches:
    raise FileNotFoundError('Attach dataset heritage-short-inputs and verify /kaggle/input mount.')
INPUT_DIR = sb_matches[0].parent
OUT_DIR = Path('/kaggle/working/outputs/shots')
OUT_DIR.mkdir(parents=True, exist_ok=True)
METRICS_PATH = Path('/kaggle/working/wan22_metrics.json')

sb = json.loads((INPUT_DIR / 'storyboard.json').read_text(encoding='utf-8'))
neg_prompt = sb['style']['negative_prompt'] + ', fast erratic motion, morphing, flickering, static frozen frame'

model_id = 'Wan-AI/Wan2.2-TI2V-5B-Diffusers'
pipe = WanImageToVideoPipeline.from_pretrained(model_id, torch_dtype=torch.float16)
pipe.enable_model_cpu_offload()
pipe.vae.enable_tiling()
print('Loaded Wan 2.2 TI2V-5B on', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
metrics = json.loads(METRICS_PATH.read_text()) if METRICS_PATH.exists() else []
# Draft at 480p (832x480 / 480x832 or square 512x512 for speed), then upgrade chosen hero shots to 720p
DRAFT_SIZE = (512, 512)
NUM_FRAMES = 49  # ~2s loopable slow motion @ 24fps (safe within T4 memory/time)

for shot in sb['shots']:
    sid = shot['id']
    out_mp4 = OUT_DIR / f'{sid}.mp4'
    if out_mp4.exists() and out_mp4.stat().st_size > 10000:
        print(f'[SKIP] {sid} already saved')
        continue
    kf_matches = sorted(INPUT_DIR.glob(f'{sid}_*.png'))
    if not kf_matches:
        print(f'[MISSING] No keyframe for {sid}')
        continue
    img = load_image(str(kf_matches[0])).resize(DRAFT_SIZE)
    gen = torch.Generator(device='cuda' if torch.cuda.is_available() else 'cpu').manual_seed(int(shot['motion']['seed']))
    t0 = time.time()
    frames = pipe(
        prompt=shot['motion']['prompt'],
        negative_prompt=neg_prompt,
        image=img,
        height=DRAFT_SIZE[1],
        width=DRAFT_SIZE[0],
        num_frames=NUM_FRAMES,
        num_inference_steps=30,
        guidance_scale=5.0,
        generator=gen,
    ).frames[0]
    export_to_video(frames, str(out_mp4), fps=24)
    elapsed_min = round((time.time() - t0) / 60.0, 3)
    metrics.append({'shot_id': sid, 'seed': shot['motion']['seed'], 'resolution': DRAFT_SIZE, 'gpu_minutes': elapsed_min})
    METRICS_PATH.write_text(json.dumps(metrics, indent=2))
    print(f'[SAVED] {out_mp4} in {elapsed_min:.2f} GPU min')